In [1]:
# M24CA1L306 Data Science Lab - Lab Exercise #23
# Student: Abhinav | MAC25MCA-2002 | S3 MCA (2025-27)
# Dataset: <PLACEHOLDER - replace with Air Quality CSV filename>
#          PLACEHOLDER: adjust column names below to match your file
# Task: DBSCAN Clustering on Air Quality Dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors


In [ ]:

# ─────────────────────────────────────────────
# Load dataset
# ─────────────────────────────────────────────
df = pd.read_csv("<AIR_QUALITY_CSV>")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())


In [ ]:

# PLACEHOLDER: drop non-numeric, datetime or ID columns
# df.drop(columns=["Date", "Time"], inplace=True, errors="ignore")

# Drop rows with missing values or fill them
df.dropna(inplace=True)

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
X = df[numeric_cols].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca   = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# ─────────────────────────────────────────────
# k-distance plot to estimate eps
# ─────────────────────────────────────────────
nbrs = NearestNeighbors(n_neighbors=5).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
distances     = np.sort(distances[:, 4])

plt.figure(figsize=(8, 4))
plt.plot(distances, color="steelblue")
plt.xlabel("Points sorted by distance")
plt.ylabel("5-NN Distance")
plt.title("k-Distance Graph (k=5) - for eps estimation")
plt.tight_layout()
plt.savefig("../Outputs/lab-23_Output/lab23_kdistance.png", dpi=100)
plt.show()

# ─────────────────────────────────────────────
# DBSCAN with default eps
# ─────────────────────────────────────────────
# PLACEHOLDER: set eps based on the elbow in k-distance graph
eps_default  = 0.5
min_samples  = 5

db     = DBSCAN(eps=eps_default, min_samples=min_samples)
labels = db.fit_predict(X_scaled)

n_clusters   = len(set(labels)) - (1 if -1 in labels else 0)
n_noise      = np.sum(labels == -1)
core_mask    = np.zeros(len(labels), dtype=bool)
core_mask[db.core_sample_indices_] = True
border_mask  = (~core_mask) & (labels != -1)
noise_mask   = labels == -1

print("\n" + "=" * 55)
print(f"Q1: Cluster Summary (eps={eps_default}, min_samples={min_samples})")
print("=" * 55)
print(f"Number of clusters formed : {n_clusters}")
print(f"Core points               : {core_mask.sum()}")
print(f"Border points             : {border_mask.sum()}")
print(f"Noise points              : {n_noise}")

# Silhouette score only if more than 1 cluster and noise not all points
if n_clusters > 1 and n_noise < len(labels):
    non_noise = labels != -1
    sil = silhouette_score(X_scaled[non_noise], labels[non_noise])
    print(f"\nQ2: Silhouette Score (excluding noise): {sil:.4f}")
else:
    print("\nQ2: Silhouette Score: not computable (only 1 cluster or all noise)")

# Cluster visualization
plt.figure(figsize=(9, 6))
unique_labels = set(labels)
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(sorted(unique_labels), colors):
    if label == -1:
        color = "black"
        marker = "x"
        lbl = "Noise"
    else:
        marker = "o"
        lbl = f"Cluster {label}"
    mask = labels == label
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=[color], marker=marker, label=lbl, alpha=0.6, s=15)

plt.title(f"DBSCAN Clusters (eps={eps_default}, min_samples={min_samples})")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(markerscale=2, fontsize=8)
plt.tight_layout()
plt.savefig("../Outputs/lab-23_Output/lab23_clusters_default.png", dpi=100)
plt.show()


In [ ]:

# ─────────────────────────────────────────────
# Q3: Parameter sensitivity - different eps values
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("Q3: Parameter Sensitivity - Different eps Values")
print("=" * 55)
print(f"{'eps':>6} | {'Clusters':>8} | {'Noise':>6} | {'Silhouette':>12}")
print("-" * 40)

eps_values = [0.3, 0.5, 1.0, 1.5, 2.0]
best_eps, best_sil = eps_default, -1

for eps in eps_values:
    db_i     = DBSCAN(eps=eps, min_samples=min_samples)
    labels_i = db_i.fit_predict(X_scaled)
    n_c      = len(set(labels_i)) - (1 if -1 in labels_i else 0)
    n_n      = np.sum(labels_i == -1)
    non_noise = labels_i != -1

    if n_c > 1 and non_noise.sum() > 1:
        sil_i = silhouette_score(X_scaled[non_noise], labels_i[non_noise])
        if sil_i > best_sil:
            best_sil = sil_i
            best_eps = eps
    else:
        sil_i = float("nan")

    print(f"{eps:>6.1f} | {n_c:>8} | {n_n:>6} | {sil_i:>12.4f}")

print(f"\nBest eps = {best_eps} with Silhouette Score = {best_sil:.4f}")

fig, axes = plt.subplots(1, len(eps_values), figsize=(20, 4))
for ax, eps in zip(axes, eps_values):
    db_i     = DBSCAN(eps=eps, min_samples=min_samples)
    labels_i = db_i.fit_predict(X_scaled)
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_i,
               cmap="tab10", s=5, alpha=0.6)
    n_c = len(set(labels_i)) - (1 if -1 in labels_i else 0)
    n_n = np.sum(labels_i == -1)
    ax.set_title(f"eps={eps}\nclusters={n_c}, noise={n_n}", fontsize=8)
    ax.set_xlabel("PC1", fontsize=7)
    ax.set_ylabel("PC2", fontsize=7)

plt.suptitle("DBSCAN Sensitivity to eps Parameter")
plt.tight_layout()
plt.savefig("../Outputs/lab-23_Output/lab23_eps_sensitivity.png", dpi=100)
plt.show()


In [ ]:

# ─────────────────────────────────────────────
# Q4: Outlier / Noise point interpretation
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("Q4: Noise Point Analysis")
print("=" * 55)

noise_points = df[numeric_cols].iloc[noise_mask]
print(f"Total noise points: {len(noise_points)}")
print("\nNoise point statistics:\n", noise_points.describe())
print("\nOverall dataset statistics:\n", df[numeric_cols].describe())

print("\nAll tasks complete. Plots saved.")